In [1]:
# All data
import pandas as pd

pd.set_option('display.precision',2)

dataframe = pd.read_csv("PPR-2019.csv")

def converter(x):
    return float(x[0].replace(',', ''))

dataframe["Price ()"] = (dataframe["Price ()"].str.split()).apply(converter)
dataframe["Price ()"] = dataframe["Price ()"].astype(float)

bypostalcode = dataframe.groupby("Postal Code").agg({"Price ()": ["mean", "count"]})
bycounty = dataframe.groupby("County").agg({"Price ()": ["mean", "count"]})

frames = [bypostalcode, bycounty]
concatdata = pd.concat(frames)
concatdata["mean_value_in_furniture"] = concatdata[("Price ()", "mean")]*0.02
concatdata["projected_value_in_furniture"] = concatdata["mean_value_in_furniture"]*concatdata[("Price ()", "count")]
sorted_values = concatdata.sort_values(by = ["projected_value_in_furniture"], ascending = False)
sorted_values["projected_value_in_furniture"] = sorted_values["projected_value_in_furniture"].apply(lambda x: '{:.2f}'.format(x))

sorted_values

Price ()        mean_value_in_furniture  \
               mean  count                           
Dublin     4.99e+05  18336                 9988.90   
Cork       2.63e+05   6514                 5265.71   
Kildare    3.04e+05   3462                 6072.09   
Meath      2.76e+05   2683                 5528.60   
Wicklow    3.54e+05   1976                 7080.08   
Galway     2.47e+05   2750                 4933.00   
Dublin 15  3.38e+05   1558                 6763.20   
Dublin 18  5.28e+05    917                10565.89   
Dublin 8   6.51e+05    676                13025.00   
Dublin 4   7.23e+05    602                14461.85   
Limerick   2.01e+05   2162                 4025.47   
Dublin 6   8.93e+05    454                17866.66   
Dublin 9   5.38e+05    746                10769.28   
Louth      2.17e+05   1719                 4339.95   
Wexford    1.89e+05   1945                 3788.63   
Dublin 14  7.50e+05    480                15000.27   
Dublin 24  3.19e+05   1102                 6380.61   
Dublin 1   1.04e+06    312                20712.73   
Waterford  1.92e+05   1569                 3844.09   
Kerry      1.85e+05   1588                 3706.80   
Dublin 16  5.56e+05    528                11119.34   
Dublin 3   6.02e+05    460                12044.96   
Dublin 13  4.57e+05    598                 9147.48   
Dublin 7   3.96e+05    667                 7926.78   
Tipperary  1.60e+05   1643                 3195.46   
Clare      1.88e+05   1229                 3760.27   
Dublin 11  2.71e+05    799                 5420.62   
Westmeath  1.81e+05   1175                 3613.92   
Dublin 12  3.30e+05    628                 6596.97   
Kilkenny   2.15e+05    924                 4301.70   
Donegal    1.27e+05   1530                 2546.91   
Dublin 2   1.03e+06    183                20651.27   
Mayo       1.41e+05   1306                 2810.89   
Dublin 5   3.97e+05    455                 7946.85   
Dublin 20  1.41e+06    122                28138.77   
Laois      1.76e+05    902                 3516.10   
Sligo      1.49e+05    864                 2980.47   
Cavan      1.46e+05    821                 2919.55   
Carlow     1.76e+05    672                 3512.05   
Offaly     1.59e+05    692                 3189.42   
Dublin 22  2.58e+05    404                 5154.32   
Roscommon  1.26e+05    806                 2518.95   
Dublin 17  6.70e+05     98                13404.94   
Monaghan   1.53e+05    388                 3059.32   
Longford   1.23e+05    451                 2466.49   
Leitrim    1.16e+05    456                 2324.85   
Dublin 10  2.12e+05    156                 4235.74   
Dublin 6w  6.73e+05     31                13455.44   

          projected_value_in_furniture  
                                        
Dublin                    183156538.25  
Cork                       34300830.41  
Kildare                    21021569.60  
Meath                      14833245.78  
Wicklow                    13990238.01  
Galway                     13565746.51  
Dublin 15                  10537060.10  
Dublin 18                   9688918.46  
Dublin 8                    8804899.41  
Dublin 4                    8706036.18  
Limerick                    8703068.77  
Dublin 6                    8111465.63  
Dublin 9                    8033883.45  
Louth                       7460372.69  
Wexford                     7368894.50  
Dublin 14                   7200131.66  
Dublin 24                   7031430.06  
Dublin 1                    6462371.79  
Waterford                   6031371.11  
Kerry                       5886397.05  
Dublin 16                   5871010.09  
Dublin 3                    5540682.18  
Dublin 13                   5470194.30  
Dublin 7                    5287164.20  
Tipperary                   5250132.92  
Clare                       4621376.32  
Dublin 11                   4331074.78  
Westmeath                   4246361.37  
Dublin 12                   4142898.11  
Kilkenny                    3974769.22 

In [33]:
import pymongo

myclient = pymongo.MongoClient("mongodb://localhost:27017/")
mydb = myclient["house_prices"]
mycollection = mydb["price_per_postcode"]


mycollection.delete_many({})
for index, row in sorted_values.iterrows():
    record = {"postcode": index, "price": row[("Price ()", "mean")], "year": "2019"}
    mycollection.insert_one(record)

